# Combat value / Critic smoke (Colab tip #2)

**Input:** `obs` vector **181-d** float32 (combat obs_v1 from tip #1). Optional semantic mapping to hp/energy/hand/intent lives in gym obs layout docs — **not** required for this smoke; HOLD JSONL is optional context only (Wiki ≠ trajectory).

**Output:** scalar **value** `(batch,)` → trained as **1-d** head.

**Label (default):** **Monte-Carlo return** — episodes split at `done=True`; backward `G_t = r_t + γ G_{t+1}` with γ=0.99; reset accumulator after terminal.

**Gate:** ≥1000 rows; finite loss; no NaN; save `.pt` checkpoint (ONNX = tip #3).

No game comms / EP / `sts2.dll`. No DT/BC batch training.

In [ ]:
# !git clone --depth 1 https://github.com/EienteiPharma/sts2-rl-agent.git /content/sts2-rl-agent
!pip install -q numpy pandas pyarrow torch

import sys
from pathlib import Path

REPO = Path("/content/sts2-rl-agent")
if not REPO.is_dir():
    raise SystemExit("Clone repo to /content/sts2-rl-agent first (uncomment git clone).")
sys.path.insert(0, str(REPO))

from sts2_env.colab.combat_critic import CriticTrainConfig, train_critic_smoke, LABEL_MC_RETURN

print("label", LABEL_MC_RETURN)

In [ ]:
from google.colab import files  # type: ignore

USE_UPLOAD = True
FEATURES_PATH = "/content/combat_features.npz"  # tip #1 export or transitions.npz
CKPT_PATH = "/content/combat_critic_smoke.pt"

if USE_UPLOAD:
    up = files.upload()
    name = next(iter(up))
    with open(FEATURES_PATH, "wb") as f:
        f.write(up[name])
    print("uploaded", name, "->", FEATURES_PATH)

# Drive: FEATURES_PATH = "/content/drive/MyDrive/sts2/combat_features.npz"

In [ ]:
meta = train_critic_smoke(
    FEATURES_PATH,
    CKPT_PATH,
    config=CriticTrainConfig(epochs=3, min_rows=1000, batch_size=256, seed=0),
)
meta

In [ ]:
import numpy as np
import torch

assert meta["n_rows"] >= 1000
assert meta["obs_dim"] == 181 and meta["out_dim"] == 1
assert all(np.isfinite(meta["losses"]))
blob = torch.load(CKPT_PATH, map_location="cpu")
assert blob["obs_dim"] == 181
print("OK smoke critic", meta)

In [ ]:
from google.colab import files  # type: ignore
files.download(CKPT_PATH)